In [21]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config.config import MODELS_BASELINE

from src.features.build_features import build_features_dataframe, feature_columns 
from src.repository.parquet_repository import ParquetRepository

from src.models.AutoThetaForecaster import AutoThetaForecaster
from src.features.features import (sort_by_forecast_date, build_date_based_cv_splits)

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:

target_col = "quantity"

LEAKAGE_SAFE_NUMERIC_FEATURES = [
    "qty_lag_origin", 
    "qty_lag_origin_7", 
    "qty_roll_mean_7",
    "qty_roll_std_7", 
    "qty_roll_mean_28", 
    "qty_roll_std_28",
    "qty_mean_ratio_7_28", 
    "qty_last_zscore_28", 
    "qty_zero_ratio_28",
    "weekday", 
    "annual_sin", 
    "annual_cos",
]

#LEAKAGE_SAFE_CATEGORICAL_FEATURES = ["Product", "Region"] nao tem

LAGGED_ANOMALY_FEATURES = [
    "anomaly_score",    
    "anomaly_score_mean7",
    "anomaly_score_mean15",
    "is_anomaly_rate7",
    "is_anomaly_rate7",
    "is_anomaly"
]

LEAKAGE_SAFE_HYBRID_NUMERIC_FEATURES = (
    LEAKAGE_SAFE_NUMERIC_FEATURES + LAGGED_ANOMALY_FEATURES
)

In [26]:
# Exemplo futuro de leitura para o LightGBM, com h=15
repo = ParquetRepository(base_path=ROOT / "data")
df = repo.load("features/anomaly_predictions.parquet")
df_15 = df[df["horizon_days"] == 7]
#df_15 = df.loc[df["horizon_days"] == 15].copy()
#df_15 = df[(df["horizon_days"] == 7) | (df["horizon_days"] == 15)] #.isin(['A', 'B', 'C'])
X_train = df_15.loc[df_15["split"] == "train", feature_columns(df_15)]
y_train = df_15.loc[df_15["split"] == "train", "quantity"]
print(X_train.shape, y_train.shape)

[OK] Arquivo carregado: d:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet
(1860, 40) (1860,)


In [27]:
df_15["date"] = pd.to_datetime(df_15["date"])

df_train_lgbm = df_15.loc[df_15["split"] == "train"].copy()
df_test_lgbm = df_15.loc[df_15["split"] == "test"].copy()

print("Treinamento:", df_train_lgbm.shape)
print("Teste:", df_test_lgbm.shape)  

Treinamento: (1860, 46)
Teste: (183, 46)


C:\Users\AlexK\AppData\Local\Temp\ipykernel_25728\1622272103.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_15["date"] = pd.to_datetime(df_15["date"])


In [28]:
model = AutoThetaForecaster(
    target_col=target_col,
    model_dir=MODELS_BASELINE,
    date_col="date",
    series_col=None,  # Uma categoria ATC por DataFrame
    forecast_horizon=15,
    n_splits=3,
    scoring="neg_root_mean_squared_error"
)

grid_result = model.gridSearchFind(df_train_lgbm)

print("Melhores parâmetros:", grid_result.best_params_)
print("Melhor score CV:", grid_result.best_score_)

model.save_best_params()
model.save_model()

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Melhores parâmetros: {'model__decomposition_type': 'additive', 'model__model': None, 'model__season_length': 7}
Melhor score CV: -11.962437224980365


WindowsPath('D:/PROJETOS/git_repo/PharmaDemand-ml/models/baseline/autotheta_model.joblib')

## Diagnóstico do intervalo de datas

A classe AutoTheta valida uma série **diária contínua**. Uma lacuna entre a data final do treino e o início do teste deve ser investigada. Não use `fillna(0)` nem interpole automaticamente vendas reais.

Como o notebook usa `anomaly_predictions.parquet` filtrado para `horizon_days == 15`, primeiro buscamos os registros originais de `quantity` nas outras linhas do próprio Parquet. Só aceitamos esses registros se houver **um único valor verdadeiro de `quantity` por data**, sem conflitos entre horizontes. Se alguma data continuar ausente, carregue a série diária original (`df_salesdaily`) em `daily_source`.

**Importante:** os dias de transição só podem ser incorporados ao histórico a partir do momento em que essas vendas já eram conhecidas. A avaliação estritamente D+15 precisa definir a origem como `data_alvo - 15 dias`.

In [29]:
# 1. Conferir a lacuna sem alterar a classe
import numpy as np
import pandas as pd

DATE = "date"
TARGET = target_col  # quantity

train_daily = df_train_lgbm[[DATE, TARGET]].copy()
test_daily = df_test_lgbm[[DATE, TARGET]].copy()
for current in (train_daily, test_daily):
    current[DATE] = pd.to_datetime(current[DATE]).dt.normalize()
    current[TARGET] = pd.to_numeric(current[TARGET], errors="raise")
    if current[DATE].duplicated().any():
        raise ValueError("Datas duplicadas: filtre uma única categoria ATC antes de usar series_col=None.")
    if current[TARGET].isna().any():
        raise ValueError("Há vendas nulas no target.")

last_train = train_daily[DATE].max()
first_test = test_daily[DATE].min()
if first_test <= last_train:
    raise ValueError("Treino e teste têm datas sobrepostas.")

all_dates = pd.DatetimeIndex(pd.concat([train_daily[DATE], test_daily[DATE]]))
expected = pd.date_range(train_daily[DATE].min(), test_daily[DATE].max(), freq="D")
missing = expected.difference(all_dates)
gap_between = pd.date_range(last_train + pd.Timedelta(days=1),
                            first_test - pd.Timedelta(days=1), freq="D")

print("Fim do treino:", last_train.date())
print("Início do teste:", first_test.date())
print("Dias ausentes entre treino/teste:", len(gap_between))
print("Ausentes em todo o período:", len(missing))
print("Exemplo:", missing[:12].date.tolist())

source_counts = df.copy()
source_counts[DATE] = pd.to_datetime(source_counts[DATE]).dt.normalize()
if "horizon_days" in source_counts.columns:
    print("Linhas candidatas por horizonte nas datas faltantes:")
    display(source_counts.loc[source_counts[DATE].isin(missing)].groupby("horizon_days").size().rename("n"))


Fim do treino: 2019-04-08
Início do teste: 2019-04-09
Dias ausentes entre treino/teste: 0
Ausentes em todo o período: 0
Exemplo: []
Linhas candidatas por horizonte nas datas faltantes:


C:\Users\AlexK\AppData\Local\Temp\ipykernel_25728\2650128258.py:26: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  gap_between = pd.date_range(last_train + pd.Timedelta(days=1),
C:\Users\AlexK\AppData\Local\Temp\ipykernel_25728\2650128258.py:27: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  first_test - pd.Timedelta(days=1), freq="D")


Series([], Name: n, dtype: int64)

## Recuperar as vendas reais faltantes

O código abaixo usa `daily_source = df` por padrão. Caso o Parquet completo não contenha todas as datas, **substitua** por um DataFrame carregado de `df_salesdaily`, já filtrado para a **mesma categoria ATC** e com colunas `date` e `quantity`. O código para com uma mensagem explícita se faltar alguma observação.

Se a sua pipeline criou um intervalo de exclusão intencional (*purge/embargo*), recuperar os valores conhecidos é aceitável para uma nova origem de previsão **após** o intervalo, mas não para fingir que eles estavam disponíveis numa origem anterior.

In [30]:
# 2. Fonte de observações REAIS; pode trocar por df_salesdaily filtrado por ATC.
daily_source = df
#daily_source = df_salesdaily


def recover_observed_days(source, dates, date_col="date", target_col="quantity"):
    required = {date_col, target_col}
    if not required.issubset(source.columns):
        raise ValueError(f"Fonte sem colunas {required - set(source.columns)}")
    obs = source[[date_col, target_col]].copy()
    obs[date_col] = pd.to_datetime(obs[date_col]).dt.normalize()
    obs[target_col] = pd.to_numeric(obs[target_col], errors="raise")
    obs = obs.loc[obs[date_col].isin(dates)]
    if obs.empty:
        raise ValueError("Nenhuma data faltante encontrada na fonte. Carregue df_salesdaily.")
    # Se o parquet tem vários horizontes, quantity deve ser a mesma venda para a mesma data.
    count_values = obs.groupby(date_col)[target_col].nunique(dropna=False)
    invalid = count_values[count_values != 1]
    if not invalid.empty:
        raise ValueError("Quantity divergente para a mesma data. "
                         "Use a série diária original e filtre uma única categoria ATC. "
                         f"Exemplos: {list(invalid.index[:4].date)}")
    bridge = obs.drop_duplicates(subset=[date_col]).sort_values(date_col).reset_index(drop=True)
    unresolved = pd.DatetimeIndex(dates).difference(pd.DatetimeIndex(bridge[date_col]))
    if len(unresolved):
        raise ValueError(f"Faltam {len(unresolved)} data(s) na fonte de vendas reais: "
                         f"{list(unresolved[:7].date)}. Carregue df_salesdaily. "
                         "Não use fillna(0).")
    if not np.isfinite(bridge[target_col].to_numpy(dtype=float)).all():
        raise ValueError("Há valores inválidos na fonte diária.")
    return bridge




In [31]:
metrics = model.evaluate_backtest(
    df_train=train_daily,
    df_test=test_daily,
    step_size=15
)

print("AutoTheta — Backtest D+1 até D+15")
for metric, value in metrics.items():
    print(f"{metric.upper():10}: {value:.4f}")

AutoTheta — Backtest D+1 até D+15
MAE       : 7.5178
RMSE      : 10.0775
R2        : 0.1626
WAPE_PCT  : 32.7853
BIAS      : -1.4244


In [32]:
bt = model.backtest(
    train_daily,
    test_daily,
    step_size=15
)

for h in [7, 15]:
    df_h = bt[bt["horizon_step"] == h]

    if len(df_h) < 2:
        continue

    y = df_h["quantity"].to_numpy()
    pred = df_h["y_pred"].to_numpy()

    print(f"\nAutoTheta D+{h}")
    print("MAE:", mean_absolute_error(y, pred))
    print("RMSE:", np.sqrt(mean_squared_error(y, pred)))
    print("R²:", r2_score(y, pred))
    print(
        "WAPE:",
        100 * np.abs(y - pred).sum() /
        np.abs(y).sum() if np.abs(y).sum() else np.nan
    )


AutoTheta D+7
MAE: 6.092369359206434
RMSE: 7.368909736247379
R²: 0.2291503031078438
WAPE: 29.691717863936326

AutoTheta D+15
MAE: 5.570197941119685
RMSE: 10.684705728443822
R²: -0.1936659471517186
WAPE: 24.65143842649317


In [ ]:
bridge = recover_observed_days(daily_source, missing, DATE, TARGET)

# Este notebook cobre a lacuna entre treino e teste. Se houver faltas DENTRO do
# teste, precisamos tratá-las separadamente, pois apenas o holdout deve ser pontuado.
if not bridge[DATE].between(last_train + pd.Timedelta(days=1),
                            first_test - pd.Timedelta(days=1)).all():
    raise ValueError("Há outras lacunas além do intervalo treino/teste. "
                     "Regularize a série diária e preserve as datas pontuadas no teste.")

train_with_bridge = pd.concat([train_daily, bridge], ignore_index=True).sort_values(DATE)
assert not train_with_bridge[DATE].duplicated().any()
assert len(pd.date_range(train_with_bridge[DATE].min(), train_with_bridge[DATE].max(), freq="D")) == len(train_with_bridge)
print("Histórico recuperado até", train_with_bridge[DATE].max().date(),
      "com", len(bridge), "observações reais adicionais.")

## Backtest em blocos de 15 dias (modelo atual)

Este é o mesmo método `backtest()` da classe, agora com histórico diário regular. **Atenção:** em cada bloco ele mede um conjunto de horizontes de D+1 a D+15, não exclusivamente previsões D+15. Portanto, não compare essa métrica diretamente com um LightGBM treinado apenas para `horizon_days == 15`.

In [ ]:
# 3A. Teste em blocos D+1..D+15 (opcional, usando a classe atual)
# Não inclui os sete dias adicionais no score: apenas os dias de df_test_lgbm.
# predictions_blocks = model.backtest(train_with_bridge, test_daily, step_size=15)
# print(predictions_blocks[[TARGET, "y_pred"]].head())
# display(predictions_blocks[[DATE, TARGET, "y_pred", "forecast_origin", "horizon_step"]].head())


## Backtest estritamente D+15 (recomendado para o stacking)

Se `horizon_days == 15` significa que a previsão para uma data `t` foi feita a partir de `t-15`, **esta** avaliação é a correta para compará-la com os regressões D+15. Para cada data-alvo do teste, refaz o ajuste com vendas observadas até a origem `t-15` e prevê exatamente o 15º dia.

A inclusão de uma venda real em treino acontece somente quando a origem temporal já a alcançou. Como são feitos vários ajustes, esta avaliação pode demorar mais que o backtest em blocos. O GridSearchCV anterior ainda usa folds de D+1..D+15; para seleção estritamente D+15, a CV também deverá ser alinhada em uma etapa posterior.

In [ ]:
# 3B. Avaliar EXATAMENTE D+15, usando os melhores parâmetros já obtidos no treino.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if model.best_params is None:
    raise ValueError("Faça gridSearchFind(df_train_lgbm) antes de executar esta célula.")

observed_all = pd.concat([train_with_bridge, test_daily], ignore_index=True).sort_values(DATE)
if observed_all[DATE].duplicated().any():
    raise ValueError("A série tem datas duplicadas.")
full_expected = pd.date_range(observed_all[DATE].min(), observed_all[DATE].max(), freq="D")
if len(full_expected) != len(observed_all):
    raise ValueError("Persistem lacunas: procure datas ausentes na fonte diária original.")

results = []
for row in test_daily.sort_values(DATE).itertuples(index=False):
    day = getattr(row, DATE)
    actual = float(getattr(row, TARGET))
    origin = pd.Timestamp(day) - pd.Timedelta(days=15)
    history = observed_all.loc[observed_all[DATE] <= origin, [DATE, TARGET]]
    if len(history) < model.min_train_days:
        raise ValueError(f"Histórico insuficiente na origem {origin.date()}.")
    # Recria o estimador com a configuração do GridSearch (sem retreinar hiperparâmetros).
    estimator = model._build_pipeline().set_params(**model.best_params)
    estimator.fit(history[[DATE]], history[TARGET].to_numpy(dtype=float))
    pred = float(estimator.predict(pd.DataFrame({DATE: [pd.Timestamp(day)]}))[0])
    results.append({DATE: pd.Timestamp(day), TARGET: actual,
                    "forecast_origin": origin, "horizon_step": 15,
                    "y_pred": pred})

predictions = pd.DataFrame(results)
y_true = predictions[TARGET].to_numpy(float)
y_pred = predictions["y_pred"].to_numpy(float)
error = y_pred - y_true
metrics = {
    "mae": float(mean_absolute_error(y_true, y_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    "r2": float(r2_score(y_true, y_pred)),
    "wape_pct": float(100 * np.abs(error).sum() / np.abs(y_true).sum())
                  if np.abs(y_true).sum() else float("nan"),
    "bias": float(error.mean()),
}
print("AutoTheta — backtest de horizonte FIXO D+15:")
print(metrics)
display(predictions.head(20))
model.save_metrics(metrics, filename="autotheta_metrics_fixed_h15.json")
